# DEST — Comprehensive: Alpha Sweep + Long Epoch + Hard Test (Kaggle, 20h)

**Skill `colab-perfecto` — 0 pip pesado, 100% reanudable**

* Fix `lexsort` 45k únicos (bug 98% apartado en `results/deprecated/`)
* Escribe en `/kaggle/working/dest_comprehensive/` (limpio)
* 1. Alpha sweep FIX: CIFAR-10 α=0.1/0.3/0.5/0.7/0.9 ×3 seeds ×2 =30 runs
* 2. Long epoch: CIFAR-10 15 vs 30ep ×3 seeds ×2 =12 runs
* 3. Hard test: CIFAR-100 α=0.5 ×3 seeds ×2 =6 runs
* Total 48 runs ~160 min T4, cabe en 20h

In [ ]:
# 0. Setup SIN pip — skill colab-perfecto
import os, sys, subprocess
print('🔧 Setup sin pip...')
if not os.path.exists('DEST'):
    subprocess.check_call(['git','clone','https://github.com/starlyn2010/DEST.git'])
if 'DEST/src' not in sys.path:
    sys.path.insert(0, 'DEST/src')
import dest
sys.modules['dest_lib']=dest
for sub in ['config','samplers','models','datasets','runner']:
    try:
        m=__import__(f'dest.{sub}', fromlist=[sub])
        sys.modules[f'dest_lib.{sub}']=m
    except Exception as e:
        print(f'warn {sub}: {e}')
print('✅ DEST fix (lexsort 45k únicos)')
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# 1. Config — 3 experimentos limpios
from dest_lib.config import get_config
import os
alphas=[0.1,0.3,0.5,0.7,0.9]
seeds=[300,301,302]
base_out='/kaggle/working/dest_comprehensive'
os.makedirs(base_out, exist_ok=True)
print(f'Output limpio: {base_out}')
print(f'Exp1 Alpha: {len(alphas)}×{len(seeds)}×2=30 runs')
print(f'Exp2 Long: 2×{len(seeds)}×2=12 runs')
print(f'Exp3 Hard: 1×{len(seeds)}×2=6 runs Total 48')


In [ ]:
# 2. Alpha sweep FIX (usa collatz_sweep)
import os, json, time, glob
from dest_lib.runner import ExperimentRunner
from dest_lib.config import get_config
for alpha in [0.1,0.3,0.5,0.7,0.9]:
    cfg=get_config('PAPER')
    cfg.update({'datasets':['CIFAR10'],'samplers':['collatz_sweep','stochastic'],'seeds':[300,301,302],'epochs':15,'batch_size':128,'lr':0.01,'lr_schedule':'cosine','val_fraction':0.1,'verbose':True,'output_dir':f'{base_out}/alpha_{alpha}'})
    runner=ExperimentRunner(cfg)
    for seed in [300,301,302]:
        for sampler_name, av in [('collatz_sweep',alpha),('stochastic',None)]:
            exp_id=f'CIFAR10_alpha{alpha}_{sampler_name}'
            out=os.path.join(cfg['output_dir'], f'{exp_id}_{sampler_name}_seed_{seed}.json')
            if os.path.exists(out):
                try:
                    j=json.load(open(out))
                    if j.get('status')=='COMPLETE' and len(j.get('test_accs',[]))==15: print(f'⏭️ {alpha} {sampler_name} {seed}'); continue
                    else: os.remove(out)
                except: os.remove(out) if os.path.exists(out) else None
            print(f'▶️ Alpha {alpha} {sampler_name} {seed}')
            kwargs={'exp_id':exp_id,'sampler_name':sampler_name,'seed':seed,'dataset':'CIFAR10'}
            if sampler_name=='collatz_sweep': kwargs['alpha_fixed']=alpha
            r=runner.run_single_seed(**kwargs)
            print(f'✅ {alpha} {sampler_name} {seed}: {r.final_test_acc:.2f}%')
    import shutil
    shutil.make_archive(f'/kaggle/working/dest_alpha_{alpha}_clean','zip',f'{base_out}/alpha_{alpha}')
    print(f'💾 alpha {alpha}')


In [ ]:
# 3. Long epoch 15 vs 30
from dest_lib.runner import ExperimentRunner
from dest_lib.config import get_config
import os, json
for epochs in [15,30]:
    cfg=get_config('PAPER')
    cfg.update({'datasets':['CIFAR10'],'samplers':['stochastic','collatz_v3'],'seeds':[300,301,302],'epochs':epochs,'batch_size':128,'lr':0.01,'lr_schedule':'cosine','val_fraction':0.1,'verbose':True,'output_dir':f'{base_out}/long_{epochs}ep'})
    runner=ExperimentRunner(cfg)
    for seed in [300,301,302]:
        for sampler_name in ['stochastic','collatz_v3']:
            exp_id=f'CIFAR10_long{epochs}_{sampler_name}'
            out=os.path.join(cfg['output_dir'], f'{exp_id}_{sampler_name}_seed_{seed}.json')
            if os.path.exists(out):
                try:
                    j=json.load(open(out))
                    if j.get('status')=='COMPLETE' and len(j.get('test_accs',[]))==epochs: print(f'⏭️ long {epochs} {sampler_name} {seed}'); continue
                    else: os.remove(out)
                except: os.remove(out) if os.path.exists(out) else None
            print(f'▶️ Long {epochs}ep {sampler_name} {seed}')
            r=runner.run_single_seed(exp_id=exp_id, sampler_name=sampler_name, seed=seed, dataset='CIFAR10')
            print(f'✅ long {epochs} {sampler_name} {seed}: {r.final_test_acc:.2f}%')
    import shutil
    shutil.make_archive(f'/kaggle/working/dest_long_{epochs}ep','zip',f'{base_out}/long_{epochs}ep')
    print(f'💾 long {epochs}ep')


In [ ]:
# 4. Hard test CIFAR-100 α=0.5
from dest_lib.runner import ExperimentRunner
from dest_lib.config import get_config
import os, json
cfg=get_config('PAPER')
cfg.update({'datasets':['CIFAR100'],'samplers':['stochastic','collatz_v3'],'seeds':[300,301,302],'epochs':15,'batch_size':128,'lr':0.01,'lr_schedule':'cosine','val_fraction':0.1,'verbose':True,'output_dir':f'{base_out}/hard_cifar100'})
runner=ExperimentRunner(cfg)
for seed in [300,301,302]:
    for sampler_name in ['stochastic','collatz_v3']:
        exp_id=f'CIFAR100_hard_{sampler_name}'
        out=os.path.join(cfg['output_dir'], f'{exp_id}_{sampler_name}_seed_{seed}.json')
        if os.path.exists(out):
            try:
                j=json.load(open(out))
                if j.get('status')=='COMPLETE' and len(j.get('test_accs',[]))==15: print(f'⏭️ hard {sampler_name} {seed}'); continue
                else: os.remove(out)
            except: os.remove(out) if os.path.exists(out) else None
        print(f'▶️ Hard {sampler_name} {seed}')
        r=runner.run_single_seed(exp_id=exp_id, sampler_name=sampler_name, seed=seed, dataset='CIFAR100')
        print(f'✅ hard {sampler_name} {seed}: {r.final_test_acc:.2f}%')
import shutil
shutil.make_archive('/kaggle/working/dest_hard_cifar100','zip',f'{base_out}/hard_cifar100')
print('💾 hard')


In [ ]:
# 5. Resumen 3 exps
import glob, json, numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
from scipy import stats
for label, pattern in [('Alpha',f'{base_out}/alpha_*/*.json'),('Long',f'{base_out}/long_*/*.json'),('Hard',f'{base_out}/hard_*/*.json')]:
    files=[f for f in glob.glob(pattern) if 'sampler_name' in json.load(open(f))]
    print(f"{label}: {len(files)} JSONs")
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j['sampler_name']].append(j['final_test_acc'])
    for s in sorted(groups):
        print(f"  {s}: {np.mean(groups[s]):.2f}±{np.std(groups[s],ddof=1):.2f}")
print('✅ resumen')


In [ ]:
# 6. Zip y descarga — todo limpio
import shutil, os, glob, json
zipname='/kaggle/working/resultados_Comprehensive_Alpha_Long_Hard'
shutil.make_archive(zipname,'zip',base_out)
print(f'✅ ZIP {zipname}.zip {os.path.getsize(zipname+".zip")/1e6:.2f} MB')
print(f'JSONs válidos: {len([f for f in glob.glob(base_out+"/**/*.json", recursive=True) if "sampler_name" in open(f).read()])} (esperado 48)')
print('Bug queda en results/deprecated/ — este zip es 100% limpio')
try:
    from google.colab import files; files.download(zipname+'.zip')
except:
    print(f'En Kaggle: Output → {zipname}.zip')
